# Heart Sound Dataset Verification, Preprocessing and Baseline Models

This notebook checks the PhysioNet heart-sound dataset structure and reference labels, inspects recordings, preprocesses audio into log-Mel features, creates train/validation/test splits, and experiments with baseline models including CNN, VGG11, BiLSTM and CRNN.

**Before running:** update `DATASET_ROOT` and other Windows paths to match your machine. Some later evaluation cells load previously saved model checkpoints, which must exist at the configured paths or those cells need to be rerun after training.

Previous execution outputs have been cleared. Run cells in order and regenerate results in your environment.

In [ ]:
import sys

print(sys.executable)


In [ ]:
from pathlib import Path

DATASET_ROOT = Path(
    r"C:\classification-of-heart-sound-recordings-the-physionetcomputing-in-cardiology-challenge-2016-1.0.0\classification-of-heart-sound-recordings-the-physionet-computing-in-cardiology-challenge-2016-1.0.0"
)

print("Dataset path:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())

In [ ]:
training_folders = [
    "training-a",
    "training-b",
    "training-c",
    "training-d",
    "training-e",
    "training-f"
]

for folder in training_folders:
    folder_path = DATASET_ROOT / folder
    print(folder, "→", folder_path.exists())

In [ ]:
training_a = DATASET_ROOT / "training-a"

print("Files in training-a:")
for file in list(training_a.iterdir())[:20]:
    print(file.name)

In [ ]:
reference_files = [
    file for file in training_a.iterdir()
    if "REFERENCE" in file.name.upper()
]

for file in reference_files:
    print(file.name)

In [ ]:
import pandas as pd

reference_path = training_a / "REFERENCE.csv"

reference = pd.read_csv(reference_path)

print(reference.head())
print()
print("Shape:", reference.shape)
print("Columns:", reference.columns.tolist())

In [ ]:
reference = pd.read_csv(
    reference_path,
    header=None,
    names=["recording", "label"]
)

print(reference.head())
print()
print("Shape:", reference.shape)
print()
print("Labels:")
print(reference["label"].value_counts())

In [ ]:
reference["wav_exists"] = reference["recording"].apply(
    lambda x: (training_a / f"{x}.wav").exists()
)

print("Total recordings:", len(reference))
print("WAV files found:", reference["wav_exists"].sum())
print("WAV files missing:", (~reference["wav_exists"]).sum())

In [ ]:
import librosa
import matplotlib.pyplot as plt

audio_path = training_a / "a0001.wav"

signal, sample_rate = librosa.load(
    audio_path,
    sr=None,
    mono=True
)

print("Sample rate:", sample_rate, "Hz")
print("Number of samples:", len(signal))
print("Duration:", len(signal) / sample_rate, "seconds")

In [ ]:
time = range(len(signal))

plt.figure(figsize=(14, 4))
plt.plot(
    [t / sample_rate for t in time],
    signal
)

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title("Raw Heart Sound Signal - a0001")
plt.show()

In [ ]:
from scipy.signal import butter, sosfiltfilt

low_cutoff = 25
high_cutoff = 400
filter_order = 5

print("Low cutoff:", low_cutoff, "Hz")
print("High cutoff:", high_cutoff, "Hz")
print("Filter order:", filter_order)

In [ ]:
from scipy.signal import butter

sos = butter(
    filter_order,
    [low_cutoff, high_cutoff],
    btype="bandpass",
    fs=sample_rate,
    output="sos"
)

print("Butterworth filter created.")
print("Filter sections:", sos.shape)

In [ ]:
from scipy.signal import sosfiltfilt

filtered_signal = sosfiltfilt(
    sos,
    signal
)

print("Filtering complete.")
print("Original samples:", len(signal))
print("Filtered samples:", len(filtered_signal))

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    [t / sample_rate for t in range(len(signal))],
    signal,
    label="Raw signal",
    alpha=0.6
)

plt.plot(
    [t / sample_rate for t in range(len(filtered_signal))],
    filtered_signal,
    label="Filtered signal"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title("Raw vs Filtered Heart Sound - a0001")
plt.legend()
plt.show()

In [ ]:
n_fft = 1024
hop_length = 256
n_mels = 128

print("FFT size:", n_fft)
print("Hop length:", hop_length)
print("Mel bands:", n_mels)

In [ ]:
mel_spectrogram = librosa.feature.melspectrogram(
    y=filtered_signal,
    sr=sample_rate,
    n_fft=n_fft,
    hop_length=hop_length,
    n_mels=n_mels
)

print("Mel spectrogram shape:", mel_spectrogram.shape)

In [ ]:
import numpy as np

In [ ]:
log_mel_spectrogram = librosa.power_to_db(
    mel_spectrogram,
    ref=np.max
)

print("Log-Mel spectrogram shape:", log_mel_spectrogram.shape)
print("Maximum value:", log_mel_spectrogram.max())
print("Minimum value:", log_mel_spectrogram.min())

In [ ]:
plt.figure(figsize=(12, 5))

librosa.display.specshow(
    log_mel_spectrogram,
    sr=sample_rate,
    hop_length=hop_length,
    x_axis="time",
    y_axis="mel"
)

plt.colorbar(format="%+2.0f dB")
plt.title("Log-Mel Spectrogram - a0001")
plt.xlabel("Time (seconds)")
plt.ylabel("Mel Frequency")
plt.show()

In [ ]:
import os

output_dir = r"C:\heart_sound_project\preprocessing\log_mel"

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "a0001.npy"
)

np.save(output_path, log_mel_spectrogram)

print("Saved:", output_path)
print("File exists:", os.path.exists(output_path))

In [ ]:
import os

log_mel_dir = r"C:\heart_sound_project\preprocessing\log_mel"

os.makedirs(log_mel_dir, exist_ok=True)

print("Output folder:", log_mel_dir)
print("Folder exists:", os.path.exists(log_mel_dir))

In [ ]:
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt
import librosa
import numpy as np
import os

# Use the training-a folder we already created
training_a_dir = training_a

# Output folder for Log-Mel features
log_mel_dir = r"C:\heart_sound_project\preprocessing\log_mel"
os.makedirs(log_mel_dir, exist_ok=True)

processed = 0
missing = 0
errors = 0

for recording_id in reference["recording"]:

    wav_path = os.path.join(
        training_a_dir,
        recording_id + ".wav"
    )

    output_path = os.path.join(
        log_mel_dir,
        recording_id + ".npy"
    )

    # Check whether WAV exists
    if not os.path.exists(wav_path):
        print("Missing:", recording_id)
        missing += 1
        continue

    try:
        # Load WAV
        sample_rate, signal = wavfile.read(wav_path)

        # Convert stereo to mono if necessary
        if signal.ndim > 1:
            signal = signal.mean(axis=1)

        # Convert to floating point
        signal = signal.astype(np.float64)

        # Butterworth band-pass filter: 25–400 Hz
        sos = butter(
            5,
            [25, 400],
            btype="bandpass",
            fs=sample_rate,
            output="sos"
        )

        filtered_signal = sosfiltfilt(
            sos,
            signal
        )

        # Generate Mel spectrogram
        mel_spectrogram = librosa.feature.melspectrogram(
            y=filtered_signal,
            sr=sample_rate,
            n_fft=1024,
            hop_length=256,
            n_mels=128
        )

        # Convert to Log-Mel
        log_mel_spectrogram = librosa.power_to_db(
            mel_spectrogram,
            ref=np.max
        )

        # Save feature
        np.save(
            output_path,
            log_mel_spectrogram
        )

        processed += 1

        if processed % 25 == 0:
            print(
                f"Processed: {processed}/{len(reference)}"
            )

    except Exception as e:
        print("Error:", recording_id, e)
        errors += 1

print("\nProcessing complete.")
print("Processed:", processed)
print("Missing:", missing)
print("Errors:", errors)

In [ ]:
from collections import Counter
import numpy as np
import os

shape_counts = Counter()

for recording_id in reference["recording"]:
    feature_path = os.path.join(
        log_mel_dir,
        recording_id + ".npy"
    )

    feature = np.load(feature_path)
    shape_counts[feature.shape] += 1

print("Number of different shapes:", len(shape_counts))
print("\nMost common shapes:")

for shape, count in shape_counts.most_common(10):
    print(shape, "→", count, "recordings")

In [ ]:
time_dimensions = [
    shape[1]
    for shape in shape_counts.keys()
]

print("Minimum time frames:", min(time_dimensions))
print("Maximum time frames:", max(time_dimensions))

In [ ]:
import numpy as np
import os

fixed_dir = r"C:\heart_sound_project\preprocessing\log_mel_fixed"
os.makedirs(fixed_dir, exist_ok=True)

TARGET_MELS = 128
TARGET_FRAMES = 286
PAD_VALUE = -80.0

processed_fixed = 0

for recording_id in reference["recording"]:

    input_path = os.path.join(
        log_mel_dir,
        recording_id + ".npy"
    )

    output_path = os.path.join(
        fixed_dir,
        recording_id + ".npy"
    )

    feature = np.load(input_path)

    # Create an empty fixed-size feature filled with -80 dB
    fixed_feature = np.full(
        (TARGET_MELS, TARGET_FRAMES),
        PAD_VALUE,
        dtype=np.float32
    )

    # Copy the actual feature into the fixed-size array
    frames_to_copy = min(
        feature.shape[1],
        TARGET_FRAMES
    )

    fixed_feature[:, :frames_to_copy] = feature[:, :frames_to_copy]

    np.save(
        output_path,
        fixed_feature
    )

    processed_fixed += 1

print("Fixed-size features created:", processed_fixed)
print("Target shape:", fixed_feature.shape)

In [ ]:
import numpy as np
import os

test_ids = ["a0001", "a0002", "a0003"]

for recording_id in test_ids:
    path = os.path.join(
        fixed_dir,
        recording_id + ".npy"
    )

    feature = np.load(path)

    print(
        recording_id,
        "→",
        feature.shape,
        "min:",
        feature.min(),
        "max:",
        feature.max()
    )

In [ ]:
import pandas as pd
import os

dataset_df = reference.copy()

dataset_df["feature_path"] = dataset_df["recording"].apply(
    lambda x: os.path.join(
        fixed_dir,
        x + ".npy"
    )
)

dataset_df["feature_exists"] = dataset_df["feature_path"].apply(
    os.path.exists
)

print(dataset_df.head())
print()
print("Total recordings:", len(dataset_df))
print("Features found:", dataset_df["feature_exists"].sum())
print("Features missing:", (~dataset_df["feature_exists"]).sum())

In [ ]:
print("Class distribution:")
print(dataset_df["label"].value_counts())

print("\nClass percentages:")
print(
    dataset_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temporary
train_df, temp_df = train_test_split(
    dataset_df,
    test_size=0.30,
    random_state=42,
    stratify=dataset_df["label"]
)

# Second split: divide the 30% temporary set equally
# → 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

In [ ]:
label_mapping = {
    -1: 0,
     1: 1
}

for df in [train_df, val_df, test_df]:
    df["target"] = df["label"].map(label_mapping)

print(train_df[["recording", "label", "target"]].head())

print("\nTrain targets:")
print(train_df["target"].value_counts())

print("\nValidation targets:")
print(val_df["target"].value_counts())

print("\nTest targets:")
print(test_df["target"].value_counts())

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class HeartSoundDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        # Load Log-Mel spectrogram
        feature = np.load(row["feature_path"])

        # Convert NumPy array to PyTorch tensor
        feature = torch.tensor(
            feature,
            dtype=torch.float32
        )

        # Add channel dimension
        feature = feature.unsqueeze(0)

        # Get target label
        target = torch.tensor(
            row["target"],
            dtype=torch.long
        )

        return feature, target

In [ ]:
train_dataset = HeartSoundDataset(train_df)
val_dataset = HeartSoundDataset(val_df)
test_dataset = HeartSoundDataset(test_df)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

feature, target = train_dataset[0]

print("\nFirst feature shape:", feature.shape)
print("First target:", target)
print("Feature dtype:", feature.dtype)
print("Target dtype:", target.dtype)

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
features, targets = next(iter(train_loader))

print("Batch feature shape:", features.shape)
print("Batch target shape:", targets.shape)
print("Targets:", targets)

In [ ]:
import torch
import torch.nn as nn

class HeartSoundCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                64 * 16 * 35,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


model = HeartSoundCNN()

print(model)

In [ ]:
features, targets = next(iter(train_loader))

outputs = model(features)

print("Input shape:", features.shape)
print("Output shape:", outputs.shape)
print("Targets shape:", targets.shape)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = HeartSoundCNN().to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)

In [ ]:
import torch

class_counts = train_df["target"].value_counts().sort_index()

total_samples = len(train_df)
num_classes = len(class_counts)

class_weights = total_samples / (
    num_classes * class_counts.values
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("Class counts:")
print(class_counts)

print("\nClass weights:")
print(class_weights)

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

print("Weighted loss:", criterion)

In [ ]:
NUM_EPOCHS = 20

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAINING
    # =========================
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(features)

        # Calculate loss
        loss = criterion(outputs, targets)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Calculate statistics
        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = model(features)

            loss = criterion(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # Save history
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    train_accuracies.append(epoch_train_accuracy)
    val_accuracies.append(epoch_val_accuracy)

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

In [ ]:
import matplotlib.pyplot as plt

# Training vs Validation Loss
plt.figure(figsize=(8, 5))

plt.plot(
    train_losses,
    label="Training Loss"
)

plt.plot(
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    train_accuracies,
    label="Training Accuracy"
)

plt.plot(
    val_accuracies,
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training and Validation Accuracy")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import numpy as np

best_epoch = np.argmax(val_accuracies) + 1
best_val_accuracy = max(val_accuracies)
best_val_loss = val_losses[best_epoch - 1]

print("Best epoch:", best_epoch)
print("Best validation accuracy:", best_val_accuracy)
print("Validation loss at best epoch:", best_val_loss)

In [ ]:
NUM_EPOCHS = 20

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

# Track the best validation performance
best_val_accuracy = 0.0
best_epoch = 0

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAINING
    # =========================
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(features)

        # Calculate weighted loss
        loss = criterion(outputs, targets)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = model(features)

            loss = criterion(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    train_accuracies.append(epoch_train_accuracy)
    val_accuracies.append(epoch_val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_val_accuracy:

        best_val_accuracy = epoch_val_accuracy
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            r"C:\heart_sound_project\models\best_cnn.pth"
        )

        print(
            f"  → Best model saved "
            f"(Epoch {best_epoch}, "
            f"Val Acc: {best_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation accuracy:", best_val_accuracy)

In [ ]:
best_cnn = HeartSoundCNN().to(device)

best_cnn.load_state_dict(
    torch.load(
        r"C:\heart_sound_project\models\best_cnn.pth",
        map_location=device
    )
)

best_cnn.eval()

print("Best CNN loaded successfully.")

In [ ]:
all_predictions = []
all_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = best_cnn(features)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            targets.cpu().numpy()
        )

print("Number of test predictions:", len(all_predictions))
print("Number of test targets:", len(all_targets))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Calculate metrics
accuracy = accuracy_score(
    all_targets,
    all_predictions
)

precision = precision_score(
    all_targets,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_targets,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_targets,
    all_predictions,
    zero_division=0
)

# Confusion matrix
cm = confusion_matrix(
    all_targets,
    all_predictions
)

print("CNN Test Results")
print("================")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

In [ ]:
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")

In [ ]:
cnn_results = {
    "Model": "CNN",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "Sensitivity": sensitivity,
    "Specificity": specificity
}

results_df = pd.DataFrame([cnn_results])

print(results_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks([0, 1], ["0", "1"])
plt.yticks([0, 1], ["0", "1"])

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

In [ ]:
import torch
import torch.nn as nn

class HeartSoundVGG11(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 8 * 17, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model_vgg = HeartSoundVGG11()

print(model_vgg)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Move VGG11 to the device
model_vgg = HeartSoundVGG11().to(device)

# Use the same class weights as the CNN
criterion_vgg = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

# New optimizer for VGG11
optimizer_vgg = torch.optim.Adam(
    model_vgg.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss function:", criterion_vgg)
print("Optimizer:", optimizer_vgg.__class__.__name__)

In [ ]:
NUM_EPOCHS_VGG = 20

vgg_train_losses = []
vgg_val_losses = []

vgg_train_accuracies = []
vgg_val_accuracies = []

best_vgg_val_accuracy = 0.0
best_vgg_epoch = 0

for epoch in range(NUM_EPOCHS_VGG):

    # =========================
    # TRAINING
    # =========================
    model_vgg.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        optimizer_vgg.zero_grad()

        outputs = model_vgg(features)

        loss = criterion_vgg(outputs, targets)

        loss.backward()

        optimizer_vgg.step()

        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    model_vgg.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = model_vgg(features)

            loss = criterion_vgg(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    vgg_train_losses.append(epoch_train_loss)
    vgg_val_losses.append(epoch_val_loss)

    vgg_train_accuracies.append(epoch_train_accuracy)
    vgg_val_accuracies.append(epoch_val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_vgg_val_accuracy:

        best_vgg_val_accuracy = epoch_val_accuracy
        best_vgg_epoch = epoch + 1

        torch.save(
            model_vgg.state_dict(),
            r"C:\heart_sound_project\models\best_vgg11.pth"
        )

        print(
            f"  → Best VGG11 saved "
            f"(Epoch {best_vgg_epoch}, "
            f"Val Acc: {best_vgg_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS_VGG}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nVGG11 Training complete.")
print("Best VGG11 epoch:", best_vgg_epoch)
print(
    "Best VGG11 validation accuracy:",
    best_vgg_val_accuracy
)

In [ ]:
best_vgg = HeartSoundVGG11().to(device)

best_vgg.load_state_dict(
    torch.load(
        r"C:\heart_sound_project\models\best_vgg11.pth",
        map_location=device
    )
)

best_vgg.eval()

print("Best VGG11 loaded successfully.")

In [ ]:
vgg_predictions = []
vgg_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = best_vgg(features)

        predictions = outputs.argmax(dim=1)

        vgg_predictions.extend(
            predictions.cpu().numpy()
        )

        vgg_targets.extend(
            targets.cpu().numpy()
        )

print("Number of VGG11 predictions:", len(vgg_predictions))
print("Number of VGG11 targets:", len(vgg_targets))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

vgg_accuracy = accuracy_score(
    vgg_targets,
    vgg_predictions
)

vgg_precision = precision_score(
    vgg_targets,
    vgg_predictions,
    zero_division=0
)

vgg_recall = recall_score(
    vgg_targets,
    vgg_predictions,
    zero_division=0
)

vgg_f1 = f1_score(
    vgg_targets,
    vgg_predictions,
    zero_division=0
)

vgg_cm = confusion_matrix(
    vgg_targets,
    vgg_predictions
)

print("VGG11 Test Results")
print("==================")
print(f"Accuracy:  {vgg_accuracy:.4f}")
print(f"Precision: {vgg_precision:.4f}")
print(f"Recall:    {vgg_recall:.4f}")
print(f"F1-score:  {vgg_f1:.4f}")

print("\nConfusion Matrix:")
print(vgg_cm)

In [ ]:
vgg_tn, vgg_fp, vgg_fn, vgg_tp = vgg_cm.ravel()

vgg_sensitivity = (
    vgg_tp / (vgg_tp + vgg_fn)
)

vgg_specificity = (
    vgg_tn / (vgg_tn + vgg_fp)
)

print(f"VGG11 Sensitivity: {vgg_sensitivity:.4f}")
print(f"VGG11 Specificity: {vgg_specificity:.4f}")

In [ ]:
vgg_results = {
    "Model": "VGG11",
    "Accuracy": vgg_accuracy,
    "Precision": vgg_precision,
    "Recall": vgg_recall,
    "F1-score": vgg_f1,
    "Sensitivity": vgg_sensitivity,
    "Specificity": vgg_specificity
}

results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([vgg_results])
    ],
    ignore_index=True
)

print(results_df)

In [ ]:
import torch
import torch.nn as nn

class HeartSoundBiLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.classifier = nn.Sequential(
            nn.Linear(64 * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        # x initially:
        # (batch, 1, 128, 286)

        # Remove channel dimension
        x = x.squeeze(1)

        # Current:
        # (batch, 128, 286)

        # Make time the sequence dimension
        x = x.transpose(1, 2)

        # Now:
        # (batch, 286, 128)

        # BiLSTM
        output, (hidden, cell) = self.lstm(x)

        # Take the final time step
        x = output[:, -1, :]

        # Classification
        x = self.classifier(x)

        return x


model_lstm = HeartSoundBiLSTM()

print(model_lstm)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Move BiLSTM to the device
model_lstm = HeartSoundBiLSTM().to(device)

# Same class weighting used for previous models
criterion_lstm = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

# New Adam optimizer
optimizer_lstm = torch.optim.Adam(
    model_lstm.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss function:", criterion_lstm)
print("Optimizer:", optimizer_lstm.__class__.__name__)

In [ ]:
NUM_EPOCHS_LSTM = 20

lstm_train_losses = []
lstm_val_losses = []

lstm_train_accuracies = []
lstm_val_accuracies = []

best_lstm_val_accuracy = 0.0
best_lstm_epoch = 0

for epoch in range(NUM_EPOCHS_LSTM):

    # =========================
    # TRAINING
    # =========================
    model_lstm.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        optimizer_lstm.zero_grad()

        outputs = model_lstm(features)

        loss = criterion_lstm(outputs, targets)

        loss.backward()

        optimizer_lstm.step()

        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    model_lstm.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = model_lstm(features)

            loss = criterion_lstm(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    lstm_train_losses.append(epoch_train_loss)
    lstm_val_losses.append(epoch_val_loss)

    lstm_train_accuracies.append(epoch_train_accuracy)
    lstm_val_accuracies.append(epoch_val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_lstm_val_accuracy:

        best_lstm_val_accuracy = epoch_val_accuracy
        best_lstm_epoch = epoch + 1

        torch.save(
            model_lstm.state_dict(),
            r"C:\heart_sound_project\models\best_bilstm.pth"
        )

        print(
            f"  → Best BiLSTM saved "
            f"(Epoch {best_lstm_epoch}, "
            f"Val Acc: {best_lstm_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS_LSTM}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nBiLSTM Training complete.")
print("Best BiLSTM epoch:", best_lstm_epoch)
print(
    "Best BiLSTM validation accuracy:",
    best_lstm_val_accuracy
)

In [ ]:
best_lstm = HeartSoundBiLSTM().to(device)

best_lstm.load_state_dict(
    torch.load(
        r"C:\heart_sound_project\models\best_bilstm.pth",
        map_location=device
    )
)

best_lstm.eval()

print("Best BiLSTM loaded successfully.")

In [ ]:
lstm_predictions = []
lstm_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = best_lstm(features)

        predictions = outputs.argmax(dim=1)

        lstm_predictions.extend(
            predictions.cpu().numpy()
        )

        lstm_targets.extend(
            targets.cpu().numpy()
        )

print(
    "Number of BiLSTM predictions:",
    len(lstm_predictions)
)

print(
    "Number of BiLSTM targets:",
    len(lstm_targets)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

lstm_accuracy = accuracy_score(
    lstm_targets,
    lstm_predictions
)

lstm_precision = precision_score(
    lstm_targets,
    lstm_predictions,
    zero_division=0
)

lstm_recall = recall_score(
    lstm_targets,
    lstm_predictions,
    zero_division=0
)

lstm_f1 = f1_score(
    lstm_targets,
    lstm_predictions,
    zero_division=0
)

lstm_cm = confusion_matrix(
    lstm_targets,
    lstm_predictions
)

print("BiLSTM Test Results")
print("===================")
print(f"Accuracy:  {lstm_accuracy:.4f}")
print(f"Precision: {lstm_precision:.4f}")
print(f"Recall:    {lstm_recall:.4f}")
print(f"F1-score:  {lstm_f1:.4f}")

print("\nConfusion Matrix:")
print(lstm_cm)

In [ ]:
lstm_tn, lstm_fp, lstm_fn, lstm_tp = lstm_cm.ravel()

lstm_sensitivity = (
    lstm_tp / (lstm_tp + lstm_fn)
)

lstm_specificity = (
    lstm_tn / (lstm_tn + lstm_fp)
)

print(f"BiLSTM Sensitivity: {lstm_sensitivity:.4f}")
print(f"BiLSTM Specificity: {lstm_specificity:.4f}")

In [ ]:
lstm_results = {
    "Model": "BiLSTM",
    "Accuracy": lstm_accuracy,
    "Precision": lstm_precision,
    "Recall": lstm_recall,
    "F1-score": lstm_f1,
    "Sensitivity": lstm_sensitivity,
    "Specificity": lstm_specificity
}

results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([lstm_results])
    ],
    ignore_index=True
)

print(results_df)

In [ ]:
import torch
import torch.nn as nn


class HeartSoundCRNN(nn.Module):

    def __init__(self):
        super().__init__()

        # =========================
        # CNN FEATURE EXTRACTOR
        # =========================
        self.cnn = nn.Sequential(

            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # =========================
        # BiLSTM
        # =========================
        self.lstm = nn.LSTM(
            input_size=64 * 16,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        # =========================
        # CLASSIFIER
        # =========================
        self.classifier = nn.Sequential(
            nn.Linear(64 * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        # Input:
        # (batch, 1, 128, 286)

        # CNN
        x = self.cnn(x)

        # After CNN:
        # (batch, 64, 16, 35)

        # Rearrange so time becomes sequence dimension
        x = x.permute(0, 3, 1, 2)

        # (batch, 35, 64, 16)

        # Combine CNN channels and frequency dimension
        x = x.reshape(
            x.size(0),
            x.size(1),
            -1
        )

        # (batch, 35, 1024)

        # BiLSTM
        x, _ = self.lstm(x)

        # Take final time step
        x = x[:, -1, :]

        # Classification
        x = self.classifier(x)

        return x


model_crnn = HeartSoundCRNN()

print(model_crnn)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Move CRNN to device
model_crnn = HeartSoundCRNN().to(device)

# Same class weights used for all previous models
criterion_crnn = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

# New optimizer for CRNN
optimizer_crnn = torch.optim.Adam(
    model_crnn.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss function:", criterion_crnn)
print("Optimizer:", optimizer_crnn.__class__.__name__)

In [ ]:
NUM_EPOCHS_CRNN = 20

crnn_train_losses = []
crnn_val_losses = []

crnn_train_accuracies = []
crnn_val_accuracies = []

best_crnn_val_accuracy = 0.0
best_crnn_epoch = 0

for epoch in range(NUM_EPOCHS_CRNN):

    # =========================
    # TRAINING
    # =========================
    model_crnn.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        optimizer_crnn.zero_grad()

        outputs = model_crnn(features)

        loss = criterion_crnn(outputs, targets)

        loss.backward()

        optimizer_crnn.step()

        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    model_crnn.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = model_crnn(features)

            loss = criterion_crnn(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    crnn_train_losses.append(epoch_train_loss)
    crnn_val_losses.append(epoch_val_loss)

    crnn_train_accuracies.append(epoch_train_accuracy)
    crnn_val_accuracies.append(epoch_val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_crnn_val_accuracy:

        best_crnn_val_accuracy = epoch_val_accuracy
        best_crnn_epoch = epoch + 1

        torch.save(
            model_crnn.state_dict(),
            r"C:\heart_sound_project\models\best_crnn.pth"
        )

        print(
            f"  → Best CRNN saved "
            f"(Epoch {best_crnn_epoch}, "
            f"Val Acc: {best_crnn_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS_CRNN}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nCRNN Training complete.")
print("Best CRNN epoch:", best_crnn_epoch)
print(
    "Best CRNN validation accuracy:",
    best_crnn_val_accuracy
)

In [ ]:
best_crnn = HeartSoundCRNN().to(device)

best_crnn.load_state_dict(
    torch.load(
        r"C:\heart_sound_project\models\best_crnn.pth",
        map_location=device
    )
)

best_crnn.eval()

print("Best CRNN loaded successfully.")

In [ ]:
crnn_predictions = []
crnn_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = best_crnn(features)

        predictions = outputs.argmax(dim=1)

        crnn_predictions.extend(
            predictions.cpu().numpy()
        )

        crnn_targets.extend(
            targets.cpu().numpy()
        )

print(
    "Number of CRNN predictions:",
    len(crnn_predictions)
)

print(
    "Number of CRNN targets:",
    len(crnn_targets)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

crnn_accuracy = accuracy_score(
    crnn_targets,
    crnn_predictions
)

crnn_precision = precision_score(
    crnn_targets,
    crnn_predictions,
    zero_division=0
)

crnn_recall = recall_score(
    crnn_targets,
    crnn_predictions,
    zero_division=0
)

crnn_f1 = f1_score(
    crnn_targets,
    crnn_predictions,
    zero_division=0
)

crnn_cm = confusion_matrix(
    crnn_targets,
    crnn_predictions
)

print("CRNN Test Results")
print("=================")
print(f"Accuracy:  {crnn_accuracy:.4f}")
print(f"Precision: {crnn_precision:.4f}")
print(f"Recall:    {crnn_recall:.4f}")
print(f"F1-score:  {crnn_f1:.4f}")

print("\nConfusion Matrix:")
print(crnn_cm)

In [ ]:
crnn_tn, crnn_fp, crnn_fn, crnn_tp = crnn_cm.ravel()

crnn_sensitivity = (
    crnn_tp / (crnn_tp + crnn_fn)
)

crnn_specificity = (
    crnn_tn / (crnn_tn + crnn_fp)
)

print(f"CRNN Sensitivity: {crnn_sensitivity:.4f}")
print(f"CRNN Specificity: {crnn_specificity:.4f}")

In [ ]:
crnn_results = {
    "Model": "CRNN",
    "Accuracy": crnn_accuracy,
    "Precision": crnn_precision,
    "Recall": crnn_recall,
    "F1-score": crnn_f1,
    "Sensitivity": crnn_sensitivity,
    "Specificity": crnn_specificity
}

results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([crnn_results])
    ],
    ignore_index=True
)

print(results_df)

In [ ]:
from pathlib import Path

training_folders = [
    "training-a",
    "training-b",
    "training-c",
    "training-d",
    "training-e",
    "training-f"
]

for folder in training_folders:

    folder_path = DATASET_ROOT / folder

    wav_files = list(folder_path.glob("*.wav"))
    dat_files = list(folder_path.glob("*.dat"))
    hea_files = list(folder_path.glob("*.hea"))

    print(f"\n{folder}")
    print("WAV files:", len(wav_files))
    print("DAT files:", len(dat_files))
    print("HEA files:", len(hea_files))

In [ ]:
reference_files_all = {}

for folder in training_folders:

    folder_path = DATASET_ROOT / folder

    reference_files = [
        file for file in folder_path.iterdir()
        if "REFERENCE" in file.name.upper()
    ]

    reference_files_all[folder] = reference_files

    print(f"\n{folder}")

    if reference_files:
        for file in reference_files:
            print("REFERENCE:", file.name)
    else:
        print("No REFERENCE file found")

In [ ]:
import pandas as pd

for folder in training_folders:

    folder_path = DATASET_ROOT / folder

    reference_file = folder_path / "REFERENCE.csv"
    sqi_file = folder_path / "REFERENCE-SQI.csv"

    print("\n" + "=" * 60)
    print(folder)

    # REFERENCE.csv
    ref_df = pd.read_csv(reference_file)

    print("\nREFERENCE.csv")
    print("Shape:", ref_df.shape)
    print("Columns:", list(ref_df.columns))
    print(ref_df.head())

    # REFERENCE-SQI.csv
    sqi_df = pd.read_csv(sqi_file)

    print("\nREFERENCE-SQI.csv")
    print("Shape:", sqi_df.shape)
    print("Columns:", list(sqi_df.columns))
    print(sqi_df.head())

In [ ]:
import pandas as pd

all_reference_data = []

for folder in training_folders:

    reference_file = DATASET_ROOT / folder / "REFERENCE.csv"

    df = pd.read_csv(
        reference_file,
        header=None,
        names=["recording", "label"]
    )

    df["folder"] = folder

    all_reference_data.append(df)


# Combine all six folders
reference_all_df = pd.concat(
    all_reference_data,
    ignore_index=True
)

print("Total recordings:", len(reference_all_df))

print("\nFirst 10 recordings:")
print(reference_all_df.head(10))

print("\nLast 10 recordings:")
print(reference_all_df.tail(10))

print("\nClass distribution:")
print(reference_all_df["label"].value_counts())

print("\nClass percentages:")
print(
    reference_all_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
reference_all_df["wav_path"] = reference_all_df.apply(
    lambda row: str(
        DATASET_ROOT
        / row["folder"]
        / f"{row['recording']}.wav"
    ),
    axis=1
)

reference_all_df["wav_exists"] = (
    reference_all_df["wav_path"]
    .apply(os.path.exists)
)

print("Total recordings:", len(reference_all_df))
print("WAV files found:", reference_all_df["wav_exists"].sum())
print("WAV files missing:", (~reference_all_df["wav_exists"]).sum())

print("\nMissing recordings:")
print(
    reference_all_df[
        ~reference_all_df["wav_exists"]
    ][["recording", "label", "folder"]]
)

In [ ]:
print("Complete dataset class distribution:")
print(reference_all_df["label"].value_counts())

print("\nComplete dataset class percentages:")
print(
    reference_all_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
from scipy.io import wavfile
from scipy.signal import butter, sosfiltfilt
import librosa
import numpy as np
import os

# Output folder for COMPLETE dataset features
log_mel_all_dir = r"C:\heart_sound_project\preprocessing\log_mel_all"

os.makedirs(log_mel_all_dir, exist_ok=True)

processed = 0
missing = 0
errors = 0

for _, row in reference_all_df.iterrows():

    recording_id = row["recording"]
    folder = row["folder"]

    wav_path = os.path.join(
        str(DATASET_ROOT),
        folder,
        recording_id + ".wav"
    )

    output_path = os.path.join(
        log_mel_all_dir,
        recording_id + ".npy"
    )

    # Check WAV
    if not os.path.exists(wav_path):
        print("Missing:", recording_id)
        missing += 1
        continue

    try:

        # =========================
        # LOAD WAV
        # =========================
        sample_rate, signal = wavfile.read(wav_path)

        # Stereo → mono
        if signal.ndim > 1:
            signal = signal.mean(axis=1)

        # Float conversion
        signal = signal.astype(np.float64)

        # =========================
        # BUTTERWORTH BAND-PASS
        # 25–400 Hz
        # =========================
        sos = butter(
            5,
            [25, 400],
            btype="bandpass",
            fs=sample_rate,
            output="sos"
        )

        filtered_signal = sosfiltfilt(
            sos,
            signal
        )

        # =========================
        # MEL SPECTROGRAM
        # =========================
        mel_spectrogram = librosa.feature.melspectrogram(
            y=filtered_signal,
            sr=sample_rate,
            n_fft=1024,
            hop_length=256,
            n_mels=128
        )

        # =========================
        # LOG-MEL
        # =========================
        log_mel_spectrogram = librosa.power_to_db(
            mel_spectrogram,
            ref=np.max
        )

        # Save as float32
        np.save(
            output_path,
            log_mel_spectrogram.astype(np.float32)
        )

        processed += 1

        if processed % 100 == 0:
            print(
                f"Processed: {processed}/"
                f"{len(reference_all_df)}"
            )

    except Exception as e:

        print(
            "Error:",
            recording_id,
            e
        )

        errors += 1


print("\n==============================")
print("Complete feature extraction")
print("==============================")
print("Processed:", processed)
print("Missing:", missing)
print("Errors:", errors)

In [ ]:
from pathlib import Path
import numpy as np
from collections import Counter

feature_dir = Path(
    r"C:\heart_sound_project\preprocessing\log_mel_all"
)

shapes = []

for file in feature_dir.glob("*.npy"):

    feature = np.load(file)

    shapes.append(feature.shape)

shape_counts = Counter(shapes)

print("Number of different shapes:", len(shape_counts))

print("\nMost common shapes:")

for shape, count in shape_counts.most_common(10):
    print(shape, "→", count, "recordings")

# Time-frame statistics
time_frames = [
    shape[1]
    for shape in shapes
]

print("\nMinimum time frames:", min(time_frames))
print("Maximum time frames:", max(time_frames))

In [ ]:
from pathlib import Path
import numpy as np

feature_dir = Path(
    r"C:\heart_sound_project\preprocessing\log_mel_all"
)

fixed_feature_dir = Path(
    r"C:\heart_sound_project\preprocessing\fixed_log_mel_all"
)

fixed_feature_dir.mkdir(parents=True, exist_ok=True)

TARGET_MELS = 128
TARGET_FRAMES = 954

processed = 0
errors = 0

for file in feature_dir.glob("*.npy"):

    try:
        feature = np.load(file)

        # Create zero-padded target array
        fixed_feature = np.zeros(
            (TARGET_MELS, TARGET_FRAMES),
            dtype=np.float32
        )

        # Number of frames available
        frames = min(
            feature.shape[1],
            TARGET_FRAMES
        )

        # Copy available frames
        fixed_feature[:, :frames] = feature[:, :frames]

        # Save fixed-size feature
        output_path = fixed_feature_dir / file.name

        np.save(
            output_path,
            fixed_feature
        )

        processed += 1

        if processed % 250 == 0:
            print(
                f"Fixed features: "
                f"{processed}/3240"
            )

    except Exception as e:

        print("Error:", file.name, e)
        errors += 1


print("\n==============================")
print("Fixed-size feature creation")
print("==============================")
print("Processed:", processed)
print("Errors:", errors)
print("Target shape:", (TARGET_MELS, TARGET_FRAMES))

In [ ]:
from pathlib import Path
import numpy as np
import os

fixed_feature_dir = Path(
    r"C:\heart_sound_project\preprocessing\fixed_log_mel_all"
)

# Add feature path to complete dataframe
reference_all_df["feature_path"] = reference_all_df[
    "recording"
].apply(
    lambda x: str(
        fixed_feature_dir / f"{x}.npy"
    )
)

# Check whether every feature exists
reference_all_df["feature_exists"] = (
    reference_all_df["feature_path"]
    .apply(os.path.exists)
)

print("Total recordings:", len(reference_all_df))

print(
    "Features found:",
    reference_all_df["feature_exists"].sum()
)

print(
    "Features missing:",
    (~reference_all_df["feature_exists"]).sum()
)

# Check first 3 feature shapes
print("\nFirst 3 features:")

for _, row in reference_all_df.head(3).iterrows():

    feature = np.load(row["feature_path"])

    print(
        row["recording"],
        "→",
        feature.shape,
        "min:",
        feature.min(),
        "max:",
        feature.max()
    )

In [ ]:
from sklearn.model_selection import train_test_split

# Convert labels:
# -1 → 0
# +1 → 1
reference_all_df["target"] = (
    reference_all_df["label"] == 1
).astype(int)

# =========================
# FIRST SPLIT
# 70% TRAIN
# 30% TEMPORARY
# =========================

train_df, temp_df = train_test_split(
    reference_all_df,
    test_size=0.30,
    stratify=reference_all_df["target"],
    random_state=42
)

# =========================
# SECOND SPLIT
# 15% VALIDATION
# 15% TEST
# =========================

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["target"],
    random_state=42
)

# Reset indexes
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# =========================
# DISPLAY SIZES
# =========================

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

# =========================
# CLASS DISTRIBUTION
# =========================

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

print("\nTrain percentages:")
print(
    train_df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


class HeartSoundDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        # Load feature
        feature = np.load(row["feature_path"])

        # Convert to float32 tensor
        feature = torch.tensor(
            feature,
            dtype=torch.float32
        )

        # Add channel dimension
        # (128, 954) → (1, 128, 954)
        feature = feature.unsqueeze(0)

        # Target
        target = torch.tensor(
            row["target"],
            dtype=torch.long
        )

        return feature, target


# Create datasets
train_dataset = HeartSoundDataset(train_df)
val_dataset = HeartSoundDataset(val_df)
test_dataset = HeartSoundDataset(test_df)


print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
features, targets = next(iter(train_loader))

print("Batch feature shape:", features.shape)
print("Batch target shape:", targets.shape)
print("Targets:", targets)

In [ ]:
import torch

# Count classes in the NEW training set
class_counts = train_df["target"].value_counts().sort_index()

print("New training class counts:")
print(class_counts)

# Calculate balanced class weights
total_samples = len(train_df)
num_classes = 2

class_weights = torch.tensor(
    [
        total_samples / (num_classes * class_counts[0]),
        total_samples / (num_classes * class_counts[1])
    ],
    dtype=torch.float32
)

print("\nNew class weights:")
print(class_weights)

In [ ]:
import torch
import torch.nn as nn


class ImprovedHeartSoundCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        # Makes output independent of the original time length
        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(128 * 4 * 4, 128),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(128, 2)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.pool(x)

        x = self.classifier(x)

        return x


model = ImprovedHeartSoundCNN()

print(model)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ImprovedHeartSoundCNN().to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)

In [ ]:
features, targets = next(iter(train_loader))

features = features.to(device)
targets = targets.to(device)

outputs = model(features)

print("Input shape:", features.shape)
print("Output shape:", outputs.shape)
print("Target shape:", targets.shape)

In [ ]:
NUM_EPOCHS = 20

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

best_val_accuracy = 0.0
best_epoch = 0

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAINING
    # =========================
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(features)

        loss = criterion(outputs, targets)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == targets
        ).sum().item()

        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = model(features)

            loss = criterion(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    train_accuracies.append(epoch_train_accuracy)
    val_accuracies.append(epoch_val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_val_accuracy:

        best_val_accuracy = epoch_val_accuracy
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            r"C:\heart_sound_project\models\best_full_cnn.pth"
        )

        print(
            f"  → Best full-dataset CNN saved "
            f"(Epoch {best_epoch}, "
            f"Val Acc: {best_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nFull-dataset CNN training complete.")
print("Best epoch:", best_epoch)
print(
    "Best validation accuracy:",
    best_val_accuracy
)

In [ ]:
# Load the best full-dataset CNN

best_full_cnn = ImprovedHeartSoundCNN().to(device)

best_full_cnn.load_state_dict(
    torch.load(
        r"C:\heart_sound_project\models\best_full_cnn.pth",
        map_location=device
    )
)

best_full_cnn.eval()

print("Best full-dataset CNN loaded successfully.")

In [ ]:
cnn_predictions = []
cnn_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = best_full_cnn(features)

        predictions = outputs.argmax(dim=1)

        cnn_predictions.extend(
            predictions.cpu().numpy()
        )

        cnn_targets.extend(
            targets.cpu().numpy()
        )

print(
    "Number of CNN predictions:",
    len(cnn_predictions)
)

print(
    "Number of CNN targets:",
    len(cnn_targets)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

cnn_accuracy = accuracy_score(
    cnn_targets,
    cnn_predictions
)

cnn_precision = precision_score(
    cnn_targets,
    cnn_predictions,
    zero_division=0
)

cnn_recall = recall_score(
    cnn_targets,
    cnn_predictions,
    zero_division=0
)

cnn_f1 = f1_score(
    cnn_targets,
    cnn_predictions,
    zero_division=0
)

cnn_cm = confusion_matrix(
    cnn_targets,
    cnn_predictions
)

print("Full-Dataset CNN Test Results")
print("=============================")
print(f"Accuracy:  {cnn_accuracy:.4f}")
print(f"Precision: {cnn_precision:.4f}")
print(f"Recall:    {cnn_recall:.4f}")
print(f"F1-score:  {cnn_f1:.4f}")

print("\nConfusion Matrix:")
print(cnn_cm)

In [ ]:
tn, fp, fn, tp = cnn_cm.ravel()

cnn_sensitivity = tp / (tp + fn)

cnn_specificity = tn / (tn + fp)

print(f"\nSensitivity: {cnn_sensitivity:.4f}")
print(f"Specificity: {cnn_specificity:.4f}")

In [ ]:
import torch
import torch.nn as nn


class ImprovedHeartSoundVGG11(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(512 * 4 * 4, 256),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(256, 2)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)

        return x


vgg11_model = ImprovedHeartSoundVGG11()

print(vgg11_model)

In [ ]:
vgg11_model = vgg11_model.to(device)

vgg11_criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

vgg11_optimizer = torch.optim.Adam(
    vgg11_model.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss function:", vgg11_criterion)
print("Optimizer:", vgg11_optimizer.__class__.__name__)

In [ ]:
features, targets = next(iter(train_loader))

features = features.to(device)
targets = targets.to(device)

outputs = vgg11_model(features)

print("Input shape:", features.shape)
print("Output shape:", outputs.shape)
print("Target shape:", targets.shape)

In [ ]:
NUM_EPOCHS = 20

vgg_train_losses = []
vgg_val_losses = []

vgg_train_accuracies = []
vgg_val_accuracies = []

best_vgg_val_accuracy = 0.0
best_vgg_epoch = 0

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAINING
    # =========================
    vgg11_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        vgg11_optimizer.zero_grad()

        outputs = vgg11_model(features)

        loss = vgg11_criterion(outputs, targets)

        loss.backward()

        vgg11_optimizer.step()

        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == targets
        ).sum().item()

        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    vgg11_model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = vgg11_model(features)

            loss = vgg11_criterion(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    vgg_train_losses.append(epoch_train_loss)
    vgg_val_losses.append(epoch_val_loss)

    vgg_train_accuracies.append(epoch_train_accuracy)
    vgg_val_accuracies.append(epoch_val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_vgg_val_accuracy:

        best_vgg_val_accuracy = epoch_val_accuracy
        best_vgg_epoch = epoch + 1

        torch.save(
            vgg11_model.state_dict(),
            r"C:\heart_sound_project\models\best_full_vgg11.pth"
        )

        print(
            f"  → Best full-dataset VGG11 saved "
            f"(Epoch {best_vgg_epoch}, "
            f"Val Acc: {best_vgg_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nFull-dataset VGG11 training complete.")
print("Best VGG11 epoch:", best_vgg_epoch)
print(
    "Best VGG11 validation accuracy:",
    best_vgg_val_accuracy
)

In [ ]:
best_full_vgg11 = ImprovedHeartSoundVGG11().to(device)

best_full_vgg11.load_state_dict(
    torch.load(
        r"C:\heart_sound_project\models\best_full_vgg11.pth",
        map_location=device
    )
)

best_full_vgg11.eval()

print("Best full-dataset VGG11 loaded successfully.")

In [ ]:
vgg_predictions = []
vgg_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = best_full_vgg11(features)

        predictions = outputs.argmax(dim=1)

        vgg_predictions.extend(
            predictions.cpu().numpy()
        )

        vgg_targets.extend(
            targets.cpu().numpy()
        )

print(
    "Number of VGG11 predictions:",
    len(vgg_predictions)
)

print(
    "Number of VGG11 targets:",
    len(vgg_targets)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

vgg_accuracy = accuracy_score(
    vgg_targets,
    vgg_predictions
)

vgg_precision = precision_score(
    vgg_targets,
    vgg_predictions,
    zero_division=0
)

vgg_recall = recall_score(
    vgg_targets,
    vgg_predictions,
    zero_division=0
)

vgg_f1 = f1_score(
    vgg_targets,
    vgg_predictions,
    zero_division=0
)

vgg_cm = confusion_matrix(
    vgg_targets,
    vgg_predictions
)

print("Full-Dataset VGG11 Test Results")
print("================================")
print(f"Accuracy:  {vgg_accuracy:.4f}")
print(f"Precision: {vgg_precision:.4f}")
print(f"Recall:    {vgg_recall:.4f}")
print(f"F1-score:  {vgg_f1:.4f}")

print("\nConfusion Matrix:")
print(vgg_cm)

In [ ]:
tn, fp, fn, tp = vgg_cm.ravel()

vgg_sensitivity = tp / (tp + fn)
vgg_specificity = tn / (tn + fp)

print(f"\nSensitivity: {vgg_sensitivity:.4f}")
print(f"Specificity: {vgg_specificity:.4f}")

In [ ]:
import torch
import torch.nn as nn


class ImprovedHeartSoundBiLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        # x: (batch, 1, 128, 954)
        x = x.squeeze(1)

        # (batch, 128, 954)
        # We want time as the sequence dimension:
        # (batch, 954, 128)
        x = x.transpose(1, 2)

        # BiLSTM
        output, _ = self.lstm(x)

        # Take the final time step
        x = output[:, -1, :]

        # Classification
        x = self.classifier(x)

        return x


bilstm_model = ImprovedHeartSoundBiLSTM()

print(bilstm_model)

In [ ]:
bilstm_model = bilstm_model.to(device)

bilstm_criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

bilstm_optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss function:", bilstm_criterion)
print(
    "Optimizer:",
    bilstm_optimizer.__class__.__name__
)

In [ ]:
features, targets = next(iter(train_loader))

features = features.to(device)
targets = targets.to(device)

outputs = bilstm_model(features)

print("Input shape:", features.shape)
print("Output shape:", outputs.shape)
print("Target shape:", targets.shape)

In [ ]:
NUM_EPOCHS = 20

bilstm_train_losses = []
bilstm_val_losses = []

bilstm_train_accuracies = []
bilstm_val_accuracies = []

best_bilstm_val_accuracy = 0.0
best_bilstm_epoch = 0

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAINING
    # =========================
    bilstm_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        bilstm_optimizer.zero_grad()

        outputs = bilstm_model(features)

        loss = bilstm_criterion(outputs, targets)

        loss.backward()

        bilstm_optimizer.step()

        running_loss += loss.item() * features.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == targets
        ).sum().item()

        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    bilstm_model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = bilstm_model(features)

            loss = bilstm_criterion(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    bilstm_train_losses.append(epoch_train_loss)
    bilstm_val_losses.append(epoch_val_loss)

    bilstm_train_accuracies.append(epoch_train_accuracy)
    bilstm_val_accuracies.append(epoch_val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_bilstm_val_accuracy:

        best_bilstm_val_accuracy = epoch_val_accuracy
        best_bilstm_epoch = epoch + 1

        torch.save(
            bilstm_model.state_dict(),
            r"C:\heart_sound_project\models\best_full_bilstm.pth"
        )

        print(
            f"  → Best full-dataset BiLSTM saved "
            f"(Epoch {best_bilstm_epoch}, "
            f"Val Acc: {best_bilstm_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nFull-dataset BiLSTM training complete.")
print("Best BiLSTM epoch:", best_bilstm_epoch)
print(
    "Best BiLSTM validation accuracy:",
    best_bilstm_val_accuracy
)

In [ ]:
best_full_bilstm = ImprovedHeartSoundBiLSTM().to(device)

best_full_bilstm.load_state_dict(
    torch.load(
        r"C:\heart_sound_project\models\best_full_bilstm.pth",
        map_location=device
    )
)

best_full_bilstm.eval()

print("Best full-dataset BiLSTM loaded successfully.")

In [ ]:
bilstm_predictions = []
bilstm_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = best_full_bilstm(features)

        predictions = outputs.argmax(dim=1)

        bilstm_predictions.extend(
            predictions.cpu().numpy()
        )

        bilstm_targets.extend(
            targets.cpu().numpy()
        )

print(
    "Number of BiLSTM predictions:",
    len(bilstm_predictions)
)

print(
    "Number of BiLSTM targets:",
    len(bilstm_targets)
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

bilstm_accuracy = accuracy_score(
    bilstm_targets,
    bilstm_predictions
)

bilstm_precision = precision_score(
    bilstm_targets,
    bilstm_predictions,
    zero_division=0
)

bilstm_recall = recall_score(
    bilstm_targets,
    bilstm_predictions,
    zero_division=0
)

bilstm_f1 = f1_score(
    bilstm_targets,
    bilstm_predictions,
    zero_division=0
)

bilstm_cm = confusion_matrix(
    bilstm_targets,
    bilstm_predictions
)

print("Full-Dataset BiLSTM Test Results")
print("================================")
print(f"Accuracy:  {bilstm_accuracy:.4f}")
print(f"Precision: {bilstm_precision:.4f}")
print(f"Recall:    {bilstm_recall:.4f}")
print(f"F1-score:  {bilstm_f1:.4f}")

print("\nConfusion Matrix:")
print(bilstm_cm)

In [ ]:
tn, fp, fn, tp = bilstm_cm.ravel()

bilstm_sensitivity = tp / (tp + fn)
bilstm_specificity = tn / (tn + fp)

print(f"\nSensitivity: {bilstm_sensitivity:.4f}")
print(f"Specificity: {bilstm_specificity:.4f}")

In [ ]:
import torch
import torch.nn as nn


class BetterHeartSoundBiLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        # (batch, 1, 128, 954)
        x = x.squeeze(1)

        # (batch, 128, 954)
        # → (batch, 954, 128)
        x = x.transpose(1, 2)

        # LSTM output
        output, _ = self.lstm(x)

        # Global temporal average instead of
        # relying only on the final time step
        x = output.mean(dim=1)

        x = self.classifier(x)

        return x


better_bilstm = BetterHeartSoundBiLSTM()

print(better_bilstm)

In [ ]:
better_bilstm = better_bilstm.to(device)

better_bilstm_criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

better_bilstm_optimizer = torch.optim.Adam(
    better_bilstm.parameters(),
    lr=0.0003
)

print("Device:", device)
print("Loss function:", better_bilstm_criterion)
print(
    "Optimizer:",
    better_bilstm_optimizer.__class__.__name__
)
print("Learning rate:", 0.0003)

In [ ]:
features, targets = next(iter(train_loader))

features = features.to(device)
targets = targets.to(device)

outputs = better_bilstm(features)

print("Input shape:", features.shape)
print("Output shape:", outputs.shape)
print("Target shape:", targets.shape)

In [ ]:
NUM_EPOCHS = 30

better_bilstm_train_losses = []
better_bilstm_val_losses = []

better_bilstm_train_accuracies = []
better_bilstm_val_accuracies = []

best_better_bilstm_val_accuracy = 0.0
best_better_bilstm_epoch = 0

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAINING
    # =========================
    better_bilstm.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        better_bilstm_optimizer.zero_grad()

        outputs = better_bilstm(features)

        loss = better_bilstm_criterion(
            outputs,
            targets
        )

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            better_bilstm.parameters(),
            max_norm=1.0
        )

        better_bilstm_optimizer.step()

        running_loss += (
            loss.item() * features.size(0)
        )

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == targets
        ).sum().item()

        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    better_bilstm.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = better_bilstm(features)

            loss = better_bilstm_criterion(
                outputs,
                targets
            )

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    better_bilstm_train_losses.append(
        epoch_train_loss
    )

    better_bilstm_val_losses.append(
        epoch_val_loss
    )

    better_bilstm_train_accuracies.append(
        epoch_train_accuracy
    )

    better_bilstm_val_accuracies.append(
        epoch_val_accuracy
    )

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_better_bilstm_val_accuracy:

        best_better_bilstm_val_accuracy = (
            epoch_val_accuracy
        )

        best_better_bilstm_epoch = epoch + 1

        torch.save(
            better_bilstm.state_dict(),
            r"C:\heart_sound_project\models\best_better_bilstm.pth"
        )

        print(
            f"  → Best improved BiLSTM saved "
            f"(Epoch {best_better_bilstm_epoch}, "
            f"Val Acc: "
            f"{best_better_bilstm_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nImproved BiLSTM training complete.")
print(
    "Best improved BiLSTM epoch:",
    best_better_bilstm_epoch
)

print(
    "Best improved BiLSTM validation accuracy:",
    best_better_bilstm_val_accuracy
)

In [ ]:
best_better_bilstm = BetterHeartSoundBiLSTM().to(device)

best_better_bilstm.load_state_dict(
    torch.load(
        r"C:\heart_sound_project\models\best_better_bilstm.pth",
        map_location=device
    )
)

best_better_bilstm.eval()

print("Best improved BiLSTM loaded successfully.")

In [ ]:
better_bilstm_predictions = []
better_bilstm_targets = []

with torch.no_grad():

    for features, targets in test_loader:

        features = features.to(device)
        targets = targets.to(device)

        outputs = best_better_bilstm(features)

        predictions = outputs.argmax(dim=1)

        better_bilstm_predictions.extend(
            predictions.cpu().numpy()
        )

        better_bilstm_targets.extend(
            targets.cpu().numpy()
        )

print(
    "Number of improved BiLSTM predictions:",
    len(better_bilstm_predictions)
)

print(
    "Number of improved BiLSTM targets:",
    len(better_bilstm_targets)
)

In [ ]:
better_bilstm_accuracy = accuracy_score(
    better_bilstm_targets,
    better_bilstm_predictions
)

better_bilstm_precision = precision_score(
    better_bilstm_targets,
    better_bilstm_predictions,
    zero_division=0
)

better_bilstm_recall = recall_score(
    better_bilstm_targets,
    better_bilstm_predictions,
    zero_division=0
)

better_bilstm_f1 = f1_score(
    better_bilstm_targets,
    better_bilstm_predictions,
    zero_division=0
)

better_bilstm_cm = confusion_matrix(
    better_bilstm_targets,
    better_bilstm_predictions
)

print("Improved BiLSTM Test Results")
print("============================")
print(f"Accuracy:  {better_bilstm_accuracy:.4f}")
print(f"Precision: {better_bilstm_precision:.4f}")
print(f"Recall:    {better_bilstm_recall:.4f}")
print(f"F1-score:  {better_bilstm_f1:.4f}")

print("\nConfusion Matrix:")
print(better_bilstm_cm)

In [ ]:
tn, fp, fn, tp = better_bilstm_cm.ravel()

better_bilstm_sensitivity = tp / (tp + fn)
better_bilstm_specificity = tn / (tn + fp)

print(
    f"\nSensitivity: {better_bilstm_sensitivity:.4f}"
)

print(
    f"Specificity: {better_bilstm_specificity:.4f}"
)

In [ ]:
import torch
import torch.nn as nn


class ImprovedHeartSoundCRNN(nn.Module):

    def __init__(self):
        super().__init__()

        # =========================
        # CNN FEATURE EXTRACTOR
        # =========================
        self.cnn = nn.Sequential(

            # Block 1
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # =========================
        # RECURRENT LAYER
        # =========================
        # After CNN:
        # 128 → 16
        # 954 → 119
        #
        # Each time step has:
        # 64 channels × 16 mel bins
        # = 1024 features
        self.lstm = nn.LSTM(
            input_size=1024,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # =========================
        # CLASSIFIER
        # =========================
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):

        # Input:
        # (batch, 1, 128, 954)

        x = self.cnn(x)

        # After CNN:
        # (batch, 64, 16, 119)

        batch_size = x.size(0)

        # Rearrange so time is the sequence dimension
        x = x.permute(0, 3, 1, 2)

        # (batch, 119, 64, 16)

        x = x.reshape(
            batch_size,
            119,
            64 * 16
        )

        # (batch, 119, 1024)

        # BiLSTM
        x, _ = self.lstm(x)

        # Average across all time steps
        x = x.mean(dim=1)

        # Classification
        x = self.classifier(x)

        return x


crnn_model = ImprovedHeartSoundCRNN()

print(crnn_model)

In [ ]:
crnn_model = crnn_model.to(device)

crnn_criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

crnn_optimizer = torch.optim.Adam(
    crnn_model.parameters(),
    lr=0.001
)

print("Device:", device)
print("Loss function:", crnn_criterion)
print(
    "Optimizer:",
    crnn_optimizer.__class__.__name__
)

In [ ]:
features, targets = next(iter(train_loader))

features = features.to(device)
targets = targets.to(device)

outputs = crnn_model(features)

print("Input shape:", features.shape)
print("Output shape:", outputs.shape)
print("Target shape:", targets.shape)

In [ ]:
NUM_EPOCHS = 20

crnn_train_losses = []
crnn_val_losses = []

crnn_train_accuracies = []
crnn_val_accuracies = []

best_crnn_val_accuracy = 0.0
best_crnn_epoch = 0

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAINING
    # =========================
    crnn_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for features, targets in train_loader:

        features = features.to(device)
        targets = targets.to(device)

        crnn_optimizer.zero_grad()

        outputs = crnn_model(features)

        loss = crnn_criterion(outputs, targets)

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            crnn_model.parameters(),
            max_norm=1.0
        )

        crnn_optimizer.step()

        running_loss += (
            loss.item() * features.size(0)
        )

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == targets
        ).sum().item()

        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_accuracy = correct / total

    # =========================
    # VALIDATION
    # =========================
    crnn_model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for features, targets in val_loader:

            features = features.to(device)
            targets = targets.to(device)

            outputs = crnn_model(features)

            loss = crnn_criterion(outputs, targets)

            val_running_loss += (
                loss.item() * features.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == targets
            ).sum().item()

            val_total += targets.size(0)

    epoch_val_loss = val_running_loss / val_total
    epoch_val_accuracy = val_correct / val_total

    # =========================
    # SAVE HISTORY
    # =========================
    crnn_train_losses.append(epoch_train_loss)
    crnn_val_losses.append(epoch_val_loss)

    crnn_train_accuracies.append(epoch_train_accuracy)
    crnn_val_accuracies.append(epoch_val_accuracy)

    # =========================
    # SAVE BEST MODEL
    # =========================
    if epoch_val_accuracy > best_crnn_val_accuracy:

        best_crnn_val_accuracy = epoch_val_accuracy
        best_crnn_epoch = epoch + 1

        torch.save(
            crnn_model.state_dict(),
            r"C:\heart_sound_project\models\best_full_crnn.pth"
        )

        print(
            f"  → Best full-dataset CRNN saved "
            f"(Epoch {best_crnn_epoch}, "
            f"Val Acc: {best_crnn_val_accuracy:.4f})"
        )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {epoch_train_loss:.4f} "
        f"Train Acc: {epoch_train_accuracy:.4f} "
        f"Val Loss: {epoch_val_loss:.4f} "
        f"Val Acc: {epoch_val_accuracy:.4f}"
    )

print("\nFull-dataset CRNN training complete.")
print("Best CRNN epoch:", best_crnn_epoch)
print(
    "Best CRNN validation accuracy:",
    best_crnn_val_accuracy
)